# 🧠 RAG Knowledge Agent & Vector Document QA (Colab T4)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SudhirDevOps1/t4-gpu-llm-api/blob/main/colab/06_rag_knowledge_agent.ipynb)
[![Lightning AI](https://img.shields.io/badge/Lightning_AI-Open_Studio-792EE5?style=for-the-badge&logo=lightning)](https://lightning.ai/new?repo_url=https://github.com/SudhirDevOps1/t4-gpu-llm-api)
[![GitHub Store](https://img.shields.io/badge/Notebook_Store-Obsidian_Hub-10b981?style=for-the-badge&logo=github)](https://github.com/SudhirDevOps1/t4-gpu-llm-api)

> **⚡ Architecture Highlights:**
> - 📚 **Multi-Format Indexing:** Ingest PDFs, Markdown, TXT, and Python source code.
> - 🔍 **High-Speed Embeddings:** Powered by ll-MiniLM-L6-v2 & Cosine Vector Indexing.
> - 🔌 **OpenAI Compatible:** Connects directly with our T4 GPU LLM Server (/v1).
> - 🛡️ **Zero-Leak Privacy:** Documents are embedded in-memory or ephemeral disk; never stored in git.


In [ ]:
#@title 🚀 1. Hardware Check & Zero-Leak Session Fingerprint
import os, sys, time, uuid, socket, hashlib

os.environ["TOKENIZERS_PARALLELISM"] = "false"
raw_fp = f"colab-rag-{socket.gethostname()}-{uuid.uuid4()}-{time.time()}"
INSTANCE_ID = "colab-rag-" + hashlib.sha256(raw_fp.encode()).hexdigest()[:8]
print(f"🆔 Unique RAG Instance Fingerprint: {INSTANCE_ID}")
print("🛡️ Privacy Status: Zero-Leak Protected")


In [ ]:
#@title 📦 2. Install Vector Database & Embedding Packages
!pip install -q chromadb sentence-transformers pypdf gradio
print("✅ ChromaDB, Sentence-Transformers & PyPDF installed successfully.")


In [ ]:
#@title 🔍 3. Launch RAG Document QA Studio (Interactive Web UI)
import gradio as gr
from sentence_transformers import SentenceTransformer
import chromadb

print("⚡ Initializing Embedding Model (all-MiniLM-L6-v2)...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
client = chromadb.Client()
collection = client.create_collection("rag_knowledge_base")

def index_text(doc_title, content):
    if not content.strip():
        return "❌ Content cannot be empty."
    # Chunking into paragraphs
    chunks = [c.strip() for c in content.split("\n\n") if c.strip()]
    embeddings = embedder.encode(chunks).tolist()
    ids = [f"{doc_title}_{i}" for i in range(len(chunks))]
    metadatas = [{"source": doc_title} for _ in range(len(chunks))]
    collection.add(ids=ids, documents=chunks, embeddings=embeddings, metadatas=metadatas)
    return f"✅ Successfully indexed {len(chunks)} knowledge chunks from '{doc_title}'!"

def query_knowledge(query_text, top_k):
    if not query_text.strip():
        return "Please enter a question."
    query_vec = embedder.encode([query_text]).tolist()
    results = collection.query(query_embeddings=query_vec, n_results=min(top_k, max(1, collection.count())))
    
    docs = results.get("documents", [[]])[0]
    if not docs:
        return "No relevant documents indexed yet. Index a document above first."
    
    context = "\n\n---\n\n".join(docs)
    rag_prompt = f"### RETRIEVED CONTEXT:\n{context}\n\n### USER QUESTION:\n{query_text}\n\n### ANSWER:"
    return rag_prompt

with gr.Blocks(title="🧠 RAG Vector Knowledge Studio") as demo:
    gr.Markdown("# 🧠 RAG Vector Knowledge Studio — Colab Free T4\nAugmented retrieval engine connecting to our OpenAI API Hub.")
    with gr.Tab("📥 Ingest Knowledge"):
        doc_title = gr.Textbox(label="Document Title", value="System Architecture Guide")
        doc_content = gr.Textbox(label="Document Text / Markdown / Code", lines=6, placeholder="Paste technical documentation, notes, or code here...")
        ingest_btn = gr.Button("Index Knowledge Document", variant="primary")
        ingest_out = gr.Textbox(label="Ingestion Status")
        ingest_btn.click(index_text, inputs=[doc_title, doc_content], outputs=[ingest_out])
    with gr.Tab("🔍 Query & Augment"):
        query_in = gr.Textbox(label="Ask Question", placeholder="What is the architecture of the system?")
        top_k = gr.Slider(minimum=1, maximum=5, value=3, step=1, label="Top-K Chunks to Retrieve")
        query_btn = gr.Button("Retrieve Relevant Knowledge", variant="primary")
        query_out = gr.Textbox(label="Augmented Prompt (Send this to your Colab LLM Studio)", lines=8)
        query_btn.click(query_knowledge, inputs=[query_in, top_k], outputs=[query_out])

demo.launch(share=True, debug=False)
